# Midterm Project — Predicción de suscripción a depósito a plazo

**Problema:** un banco portugués realiza campañas telefónicas para ofrecer un
depósito a plazo. Llamar a todos los clientes es caro; el objetivo es
**priorizar las llamadas** prediciendo la probabilidad de que un cliente
suscriba el producto (`y = yes`).

**Dataset:** [UCI Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing)
(`bank-full.csv`, 45.211 filas, 16 features + target). Licencia CC BY 4.0
(Moro, Rita & Cortez, 2014).

**Métrica principal:** ROC-AUC, porque el target está desbalanceado
(~11,7% de positivos). También se reportan PR-AUC, precision, recall y F1, y se
elige un umbral de decisión por F1.

**Nota sobre leakage:** se elimina `duration` (duración de la última llamada),
porque solo se conoce *después* de la llamada y no está disponible antes de
decidir a quién llamar.

## 1. Imports y configuración

In [1]:
import hashlib
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from xgboost import XGBClassifier

from pipeline import (
    CATEGORICAL_FEATURES,
    FEATURE_COLUMNS,
    NUMERICAL_FEATURES,
    build_pipeline,
    engineer_features,
    load_features_and_target,
)

RANDOM_STATE = 42
DATA_PATH = Path("data/bank-full.csv")
IMAGES = Path("images")
IMAGES.mkdir(exist_ok=True)

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.max_columns", 50)

print("dataset sha256:", hashlib.sha256(DATA_PATH.read_bytes()).hexdigest())

dataset sha256: d1513ec63b385506f7cfce9f2c5caa9fe99e7ba4e8c3fa264b3aaf0f849ed32d


## 2. Carga y revisión inicial

In [2]:
raw = pd.read_csv(DATA_PATH, sep=";")
print("shape:", raw.shape)
print("columnas:", raw.columns.tolist())
raw.head()

shape: (45211, 17)
columnas: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [3]:
print("valores nulos por columna:")
print(raw.isna().sum())
print("\nfilas duplicadas:", raw.duplicated().sum())
raw.dtypes.to_frame("dtype")

valores nulos por columna:
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

filas duplicadas: 0


,dtype
age,int64
job,object
marital,object
education,object
default,object
balance,int64
housing,object
loan,object
contact,object
day,int64


In [4]:
raw.describe().T

,count,mean,std,min,25%,50%,75%,max
age,45211.0,40.936210,10.618762,18.0,33.0,39.0,48.0,95.0
balance,45211.0,1362.272058,3044.765829,-8019.0,72.0,448.0,1428.0,102127.0
day,45211.0,15.806419,8.322476,1.0,8.0,16.0,21.0,31.0
duration,45211.0,258.163080,257.527812,0.0,103.0,180.0,319.0,4918.0
campaign,45211.0,2.763841,3.098021,1.0,1.0,2.0,3.0,63.0
pdays,45211.0,40.197828,100.128746,-1.0,-1.0,-1.0,-1.0,871.0
previous,45211.0,0.580323,2.303441,0.0,0.0,0.0,0.0,275.0


## 3. Variable objetivo

El target está desbalanceado: la mayoría de los clientes **no** suscribe.
Por eso `accuracy` es engañosa (un modelo que siempre dice "no" acierta ~88%)
y se usa **ROC-AUC** como métrica principal.

In [5]:
target_counts = raw["y"].value_counts()
target_rates = raw["y"].value_counts(normalize=True).round(4)
print(target_counts)
print(target_rates)

fig, ax = plt.subplots()
target_counts.plot(kind="bar", color=["#4C72B0", "#DD8452"], ax=ax)
ax.set_title("Distribución del target")
ax.set_xlabel("y")
ax.set_ylabel("clientes")
for i, value in enumerate(target_counts.values):
    ax.text(i, value, f"{value:,}", ha="center", va="bottom")
fig.tight_layout()
fig.savefig(IMAGES / "target_distribution.png", dpi=150)
plt.show()

y
no     39922
yes     5289
Name: count, dtype: int64
y
no     0.883
yes    0.117
Name: proportion, dtype: float64


## 4. EDA de variables categóricas

Se calcula la tasa de conversión (`yes`) por categoría. Esto muestra qué
segmentos responden mejor y alimenta la narrativa de negocio.

In [6]:
categorical_raw = raw.select_dtypes(include="object").columns.drop("y")
for column in categorical_raw:
    rates = (
        raw.groupby(column)["y"]
        .apply(lambda series: (series == "yes").mean())
        .sort_values(ascending=False)
    )
    print(f"\n=== {column} ===")
    print(rates.round(3).to_string())


=== job ===
job
student          0.287
retired          0.228
unemployed       0.155
management       0.138
admin.           0.122
self-employed    0.118
unknown          0.118
technician       0.111
services         0.089
housemaid        0.088
entrepreneur     0.083
blue-collar      0.073

=== marital ===
marital
single      0.149
divorced    0.119
married     0.101

=== education ===
education
tertiary     0.150
unknown      0.136
secondary    0.106
primary      0.086

=== default ===
default
no     0.118
yes    0.064

=== housing ===
housing
no     0.167
yes    0.077

=== loan ===
loan
no     0.127
yes    0.067

=== contact ===
contact
cellular     0.149
telephone    0.134
unknown      0.041

=== month ===
month
mar    0.520
dec    0.467
sep    0.465
oct    0.438
apr    0.197
feb    0.166
aug    0.110
jun    0.102
nov    0.102
jan    0.101
jul    0.091
may    0.067

=== poutcome ===
poutcome
success    0.647
other      0.167
failure    0.126
unknown    0.092


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, column in zip(axes, ["month", "poutcome"]):
    rates = (
        raw.groupby(column)["y"]
        .apply(lambda series: (series == "yes").mean())
        .sort_values(ascending=False)
    )
    rates.plot(kind="bar", ax=ax, color="#55A868")
    ax.set_title(f"Tasa de conversión por {column}")
    ax.set_ylabel("P(y = yes)")
    ax.tick_params(axis="x", rotation=45)

fig.tight_layout()
fig.savefig(IMAGES / "conversion_by_month_poutcome.png", dpi=150)
plt.show()

In [8]:
fig, ax = plt.subplots(figsize=(9, 4))
rates = (
    raw.groupby("job")["y"]
    .apply(lambda series: (series == "yes").mean())
    .sort_values(ascending=False)
)
rates.plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_title("Tasa de conversión por tipo de trabajo")
ax.set_ylabel("P(y = yes)")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(IMAGES / "conversion_by_job.png", dpi=150)
plt.show()

## 5. EDA de variables numéricas

Se comparan las distribuciones por clase y se revisan correlaciones entre
variables numéricas.

In [9]:
raw.groupby("y")[["age", "balance", "day", "campaign", "pdays", "previous"]].describe().T

y                          no           yes
age      count   39922.000000   5289.000000
         mean       40.838986     41.670070
         std        10.172662     13.497781
         min        18.000000     18.000000
         25%        33.000000     31.000000
         50%        39.000000     38.000000
         75%        48.000000     50.000000
         max        95.000000     95.000000
balance  count   39922.000000   5289.000000
         mean     1303.714969   1804.267915
         std      2974.195473   3501.104777
         min     -8019.000000  -3058.000000
         25%        58.000000    210.000000
         50%       417.000000    733.000000
         75%      1345.000000   2159.000000
         max    102127.000000  81204.000000
day      count   39922.000000   5289.000000
         mean       15.892290     15.158253
         std         8.294728      8.501875
         min         1.000000      1.000000
         25%         8.000000      8.000000
         50%        16.000000     15.000000
         75%        21.000000     22.000000
         max        31.000000     31.000000
campaign count   39922.000000   5289.000000
         mean        2.846350      2.141047
         std         3.212767      1.921826
         min         1.000000      1.000000
         25%         1.000000      1.000000
         50%         2.000000      2.000000
         75%         3.000000      3.000000
         max        63.000000     32.000000
pdays    count   39922.000000   5289.000000
         mean       36.421372     68.702968
         std        96.757135    118.822266
         min        -1.000000     -1.000000
         25%        -1.000000     -1.000000
         50%        -1.000000     -1.000000
         75%        -1.000000     98.000000
         max       871.000000    854.000000
previous count   39922.000000   5289.000000
         mean        0.502154      1.170354
         std         2.256771      2.553272
         min         0.000000      0.000000
         25%         0.000000      0.000000
         50%         0.000000      0.000000
         75%         0.000000      1.000000
         max       275.000000     58.000000

In [10]:
numeric_raw = ["age", "balance", "day", "campaign", "pdays", "previous"]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, column in zip(axes.ravel(), numeric_raw):
    data = [raw.loc[raw.y == "no", column], raw.loc[raw.y == "yes", column]]
    ax.boxplot(data, tick_labels=["no", "yes"], showfliers=False)
    ax.set_title(column)
fig.tight_layout()
fig.savefig(IMAGES / "numeric_by_target.png", dpi=150)
plt.show()

In [11]:
corr = raw[["age", "balance", "day", "duration", "campaign", "pdays", "previous"]].corr()
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax)
ax.set_title("Correlación de variables numéricas")
fig.tight_layout()
fig.savefig(IMAGES / "correlation_heatmap.png", dpi=150)
plt.show()

## 6. Preparación de datos

- `was_contacted_before = (pdays != -1)`: en este dataset `pdays = -1` significa
  que el cliente no había sido contactado antes.
- Se elimina `duration` por leakage.
- Split estratificado **60% train / 20% validation / 20% test**.
- Preprocesamiento: `StandardScaler` para numéricas y `OneHotEncoder` para
  categóricas, dentro de un `Pipeline` (evita leakage y se reutiliza en
  `train.py` y `predict.py`).

In [12]:
features, target, _ = load_features_and_target(DATA_PATH)
print("features:", features.shape)
print("columnas usadas:", FEATURE_COLUMNS)
print("tasa de positivos:", round(target.mean(), 4))

X_full_train, X_test, y_full_train, y_test = train_test_split(
    features, target, test_size=0.2, stratify=target, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_full_train, y_full_train, test_size=0.25, stratify=y_full_train,
    random_state=RANDOM_STATE,
)

print("train / val / test:", len(X_train), len(X_val), len(X_test))
print("positivos train/val/test:",
      round(y_train.mean(), 4), round(y_val.mean(), 4), round(y_test.mean(), 4))

features: (45211, 16)
columnas usadas: ['age', 'balance', 'day', 'campaign', 'pdays', 'previous', 'was_contacted_before', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
tasa de positivos: 0.117
train / val / test: 27126 9042 9043
positivos train/val/test: 0.117 0.117 0.117


## 7. Modelos y tuning

Se comparan cuatro enfoques con validación cruzada estratificada de 5 folds
sobre el conjunto de entrenamiento:

1. `DummyClassifier` (baseline trivial).
2. Regresión logística (tuning de `C`).
3. Random Forest (tuning de `max_depth` y `min_samples_leaf`).
4. XGBoost (tuning de `learning_rate`, `max_depth` y `min_child_weight`).

La métrica de búsqueda es ROC-AUC.

In [13]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

model_grids = {
    "dummy": (
        DummyClassifier(strategy="prior"),
        {},
    ),
    "logistic_regression": (
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        {"model__C": [0.01, 0.1, 1.0, 10.0]},
    ),
    "random_forest": (
        RandomForestClassifier(
            n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
        ),
        {
            "model__max_depth": [10, 20, None],
            "model__min_samples_leaf": [1, 5],
        },
    ),
    "xgboost": (
        XGBClassifier(
            n_estimators=300,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="auc",
            tree_method="hist",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        {
            "model__learning_rate": [0.05, 0.1],
            "model__max_depth": [4, 6],
            "model__min_child_weight": [1, 5],
        },
    ),
}

searches = {}
results = []

for name, (classifier, grid) in model_grids.items():
    pipe = build_pipeline(classifier)
    search = GridSearchCV(
        pipe, grid, scoring="roc_auc", cv=cv, n_jobs=1, refit=True
    )
    search.fit(X_train, y_train)
    val_prob = search.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_prob)
    val_f1 = f1_score(y_val, (val_prob >= 0.5).astype(int))

    searches[name] = search
    results.append(
        {
            "model": name,
            "cv_auc_mean": search.best_score_,
            "cv_auc_std": search.cv_results_["std_test_score"][search.best_index_],
            "val_auc": val_auc,
            "val_f1_0.5": val_f1,
            "best_params": search.best_params_,
        }
    )
    print(f"{name:20s} cv_auc={search.best_score_:.4f} val_auc={val_auc:.4f} val_f1={val_f1:.4f}")

results_df = pd.DataFrame(results).sort_values("val_auc", ascending=False)
results_df

dummy                cv_auc=0.5000 val_auc=0.5000 val_f1=0.0000


logistic_regression  cv_auc=0.7613 val_auc=0.7702 val_f1=0.2808


random_forest        cv_auc=0.7904 val_auc=0.8013 val_f1=0.3084


xgboost              cv_auc=0.7959 val_auc=0.8088 val_f1=0.3660


,model,cv_auc_mean,cv_auc_std,val_auc,val_f1_0.5,best_params
3,xgboost,0.795906,0.008315,0.808826,0.366004,"{'model__learning_rate': 0.05, 'model__max_dep..."
2,random_forest,0.790435,0.008367,0.801312,0.308377,"{'model__max_depth': 20, 'model__min_samples_l..."
1,logistic_regression,0.761332,0.007428,0.770208,0.280781,{'model__C': 0.1}
0,dummy,0.500000,0.000000,0.500000,0.000000,{}


## 8. Selección del mejor modelo y umbral

In [14]:
candidates = results_df[results_df.model != "dummy"]
best_row = candidates.iloc[0]
best_name = best_row.model
best_search = searches[best_name]
best_estimator = best_search.best_estimator_

print("Mejor modelo:", best_name)
print("Mejores hiperparámetros:", best_search.best_params_)
print(f"CV AUC: {best_row.cv_auc_mean:.4f} +/- {best_row.cv_auc_std:.4f}")
print(f"Validation AUC: {best_row.val_auc:.4f}")

val_prob = best_estimator.predict_proba(X_val)[:, 1]
thresholds = np.round(np.arange(0.05, 0.96, 0.01), 2)
f1_scores = [f1_score(y_val, (val_prob >= t).astype(int)) for t in thresholds]
best_threshold = float(thresholds[int(np.argmax(f1_scores))])

print(f"Umbral óptimo por F1 en validación: {best_threshold:.2f}")
print(f"F1 en validación con ese umbral: {max(f1_scores):.4f}")

fig, ax = plt.subplots()
ax.plot(thresholds, f1_scores, color="#C44E52")
ax.axvline(best_threshold, linestyle="--", color="black",
           label=f"mejor umbral = {best_threshold:.2f}")
ax.set_xlabel("Umbral")
ax.set_ylabel("F1 (validación)")
ax.set_title("F1 vs umbral de decisión")
ax.legend()
fig.tight_layout()
fig.savefig(IMAGES / "threshold_f1.png", dpi=150)
plt.show()

Mejor modelo: xgboost
Mejores hiperparámetros: {'model__learning_rate': 0.05, 'model__max_depth': 4, 'model__min_child_weight': 1}
CV AUC: 0.7959 +/- 0.0083
Validation AUC: 0.8088


Umbral óptimo por F1 en validación: 0.23
F1 en validación con ese umbral: 0.5078


## 9. Evaluación final en test

Se reentrena el mejor modelo con **train + validation** y se evalúa una sola
vez en el conjunto de test (20% no visto).

In [15]:
final_model = clone(best_estimator)
final_model.fit(X_full_train, y_full_train)
test_prob = final_model.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, test_prob)
test_pr_auc = average_precision_score(y_test, test_prob)

test_metrics = {"auc": test_auc, "pr_auc": test_pr_auc}

for label, threshold in [("0.50", 0.5), (f"F1-opt ({best_threshold:.2f})", best_threshold)]:
    prediction = (test_prob >= threshold).astype(int)
    metrics = {
        "precision": precision_score(y_test, prediction),
        "recall": recall_score(y_test, prediction),
        "f1": f1_score(y_test, prediction),
        "confusion_matrix": confusion_matrix(y_test, prediction).tolist(),
    }
    test_metrics[label] = metrics
    print(f"\nUmbral {label}")
    print(f"  precision = {metrics['precision']:.4f}")
    print(f"  recall    = {metrics['recall']:.4f}")
    print(f"  f1        = {metrics['f1']:.4f}")
    print(f"  matriz de confusión = {metrics['confusion_matrix']}")

print(f"\nTest ROC-AUC = {test_auc:.4f}")
print(f"Test PR-AUC  = {test_pr_auc:.4f}")


Umbral 0.50
  precision = 0.6718
  recall    = 0.2476
  f1        = 0.3619
  matriz de confusión = [[7857, 128], [796, 262]]

Umbral F1-opt (0.23)
  precision = 0.4981
  recall    = 0.5019
  f1        = 0.5000
  matriz de confusión = [[7450, 535], [527, 531]]

Test ROC-AUC = 0.8069
Test PR-AUC  = 0.4719


## 10. Importancia de features

Se usa *permutation importance* sobre el modelo entrenado con train y el
conjunto de validación: mide cuánto cae el AUC al permutar cada feature
original, sin depender de la escala ni del one-hot encoding.

In [16]:
perm = permutation_importance(
    best_estimator,
    X_val,
    y_val,
    scoring="roc_auc",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=1,
)
importance = (
    pd.DataFrame(
        {
            "feature": X_val.columns,
            "importance_mean": perm.importances_mean,
            "importance_std": perm.importances_std,
        }
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)
importance

,feature,importance_mean,importance_std
0,month,0.067025,0.003705
1,contact,0.057588,0.004103
2,day,0.028487,0.002775
3,poutcome,0.016975,0.001914
4,housing,0.009545,0.001165
5,pdays,0.008184,0.001113
6,marital,0.006846,0.001538
7,balance,0.006184,0.001525
8,age,0.005866,0.002022
9,campaign,0.002479,0.001007


In [17]:
fig, ax = plt.subplots(figsize=(8, 6))
ordered = importance.sort_values("importance_mean")
ax.barh(ordered.feature, ordered.importance_mean, xerr=ordered.importance_std,
        color="#4C72B0", alpha=0.9)
ax.set_xlabel("Caída media de AUC al permutar")
ax.set_title(f"Permutation importance ({best_name})")
fig.tight_layout()
fig.savefig(IMAGES / "feature_importance.png", dpi=150)
plt.show()

## 11. Sanity check: split temporal

`bank-full.csv` está ordenado por fecha (mayo 2008 – noviembre 2010). El split
aleatorio puede ser optimista. Como chequeo de robustez, se entrena con el 80%
más antiguo y se evalúa en el 20% más reciente.

In [18]:
n_train = int(len(features) * 0.8)
temporal_model = clone(best_estimator)
temporal_model.fit(features.iloc[:n_train], target.iloc[:n_train])
temporal_prob = temporal_model.predict_proba(features.iloc[n_train:])[:, 1]
temporal_auc = roc_auc_score(target.iloc[n_train:], temporal_prob)

print("Split temporal (80% antiguo -> 20% reciente)")
print(f"  train: {n_train:,} filas | test: {len(features) - n_train:,} filas")
print(f"  tasa de positivos train: {target.iloc[:n_train].mean():.4f}")
print(f"  tasa de positivos test : {target.iloc[n_train:].mean():.4f}")
print(f"  ROC-AUC temporal       : {temporal_auc:.4f}")

Split temporal (80% antiguo -> 20% reciente)
  train: 36,168 filas | test: 9,043 filas
  tasa de positivos train: 0.0673
  tasa de positivos test : 0.3158
  ROC-AUC temporal       : 0.6851


## 12. Conclusiones

- El target está desbalanceado (~11,7% positivos); `accuracy` no es útil y se
  optimiza **ROC-AUC**.
- Se descartó `duration` para evitar leakage: el modelo usa solo información
  disponible **antes** de la llamada.
- `pdays = -1` se tradujo a `was_contacted_before`, una señal más interpretable.
- Los modelos de árboles/boosting superan a la regresión logística; el mejor se
  elige por AUC de validación y se evalúa una sola vez en test.
- El umbral se ajusta por F1 en validación para equilibrar precision y recall
  según el costo de llamar a un cliente que no comprará.
- El split temporal da una estimación más conservadora y realista.
- Próximos pasos: calibrar probabilidades, optimizar la ganancia esperada
  (costo de llamada vs valor del depósito) y monitorear drift.

In [19]:
summary = {
    "best_model": best_name,
    "best_params": {k: str(v) for k, v in best_search.best_params_.items()},
    "cv_auc_mean": float(best_row.cv_auc_mean),
    "cv_auc_std": float(best_row.cv_auc_std),
    "validation_auc": float(best_row.val_auc),
    "threshold": best_threshold,
    "test_auc": float(test_auc),
    "test_pr_auc": float(test_pr_auc),
    "temporal_auc": float(temporal_auc),
}
print(json.dumps(summary, indent=2))
Path("images/metrics_summary.json").write_text(json.dumps(summary, indent=2) + "\n")

{
  "best_model": "xgboost",
  "best_params": {
    "model__learning_rate": "0.05",
    "model__max_depth": "4",
    "model__min_child_weight": "1"
  },
  "cv_auc_mean": 0.7959064752517577,
  "cv_auc_std": 0.008315231842443847,
  "validation_auc": 0.8088259458425358,
  "threshold": 0.23,
  "test_auc": 0.8069407075885432,
  "test_pr_auc": 0.47188242832258304,
  "temporal_auc": 0.6850763539616591
}


400